# vi-function-calling-slm · 03 DPO trên model SFT

Cần **GPU T4** (Unsloth không chạy trên P100). Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Input: code + output notebook 01 (dữ liệu) + output notebook 02 (`vi-fc-sft-lora`); trên Colab tất cả nằm
sẵn trên Drive. Secret: `HF_TOKEN`. Colab bị ngắt thì chạy lại notebook: bản merge SFT, lỗi on-policy và
checkpoint DPO (`outputs/dpo`) đều lưu trên Drive nên không phải làm lại từ đầu.

**Bẫy model tham chiếu:** DPOTrainer với PEFT và `ref_model=None` lấy model *tắt adapter* làm tham chiếu.
Nếu cứ load adapter SFT rồi train tiếp thì tắt adapter = Qwen3 gốc, không phải model SFT -> DPO kéo về sai mốc.
Nên: gộp adapter SFT vào trọng số (merge 16-bit) -> load lại 4-bit -> gắn LoRA mới cho DPO.
Khi đó tắt adapter = đúng model SFT.

Cặp preference: nhiễu có chủ đích (notebook 01) + lỗi thật của model SFT trên một phần tập train (on-policy).
beta = 0.1, lr 5e-6, 1 epoch.

In [ ]:
import gc, glob, json, os, random, sys

IN_COLAB = "google.colab" in sys.modules
# Colab bản mới cũng có sẵn /kaggle/input (tích hợp dataset Kaggle) -> nhận Kaggle bằng biến môi trường riêng của nó
IN_KAGGLE = not IN_COLAB and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/vi_fc/__init__.py", recursive=True)[0].rsplit("/vi_fc", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/vi-function-calling-slm" if IN_COLAB else os.path.abspath("vi-function-calling-slm")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/vi-function-calling-slm {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/vi-function-calling-slm" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)


def find(pat, hint):
    # output của notebook trước: Kaggle thì add làm input (/kaggle/input), Colab thì nằm sẵn trong WORK trên Drive
    for root in (["/kaggle/input"] if IN_KAGGLE else []) + [WORK, REPO]:
        hits = glob.glob(f"{root}/{pat}") or glob.glob(f"{root}/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"không tìm thấy {pat}: {hint}")


DATA = os.path.dirname(find("gen/train.jsonl", "chạy notebook 01 trước"))
SFT_DIR = os.path.dirname(find("vi-fc-sft-lora/adapter_config.json", "chạy notebook 02 trước"))
print(CODE, DATA, SFT_DIR)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q unsloth

token = secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
else:
    print("thiếu HF_TOKEN, sẽ không push")

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer

PatchDPOTrainer()
import torch

MAX_LEN = 3072
# 1) base + adapter SFT ở fp16 -> merge (Colab chạy lại thì bản merge đã có trên Drive)
if not os.path.exists("sft-merged/config.json"):
    model, tokenizer = FastLanguageModel.from_pretrained(SFT_DIR, max_seq_length=MAX_LEN, load_in_4bit=False,
                                                         dtype=torch.float16)
    model.save_pretrained_merged("sft-merged", tokenizer, save_method="merged_16bit")
    del model
    gc.collect(); torch.cuda.empty_cache()

# 2) load lại bản merge ở 4-bit
model, tokenizer = FastLanguageModel.from_pretrained("sft-merged", max_seq_length=MAX_LEN, load_in_4bit=True)

## Lỗi on-policy: chạy model SFT trên 600 câu train, giữ câu sai làm rejected

In [ ]:
from vi_fc.dpo import mine_on_policy
from vi_fc.evaluate import compute_metrics, load_jsonl, predict
from vi_fc.inference import FunctionCaller

train = load_jsonl(f"{DATA}/train.jsonl")
sub = random.Random(0).sample(train, min(600, len(train)))
# lưu ra file: Colab bị ngắt thì lần sau đọc lại, cặp DPO giữ nguyên để resume checkpoint cho đúng
rows = load_jsonl("sft_train600.jsonl") if os.path.exists("sft_train600.jsonl") else []
if len(rows) < len(sub):
    FastLanguageModel.for_inference(model)
    fc = FunctionCaller("hf", model, tokenizer=tokenizer, guard=False)
    rows = predict(sub, fc, "sft", "train600", "sft_train600.jsonl")
print("SFT trên train600:", {k: v for k, v in compute_metrics(rows).items() if k != "exact_by_scenario"})
on_policy = mine_on_policy(sub, rows)  # rows có cả raw: output hỏng format giữ nguyên làm rejected
print(len(on_policy), "cặp on-policy")

In [ ]:
programmatic = load_jsonl(f"{DATA}/dpo_train.jsonl")
random.Random(0).shuffle(programmatic)
pairs = on_policy + programmatic[:2000]
from collections import Counter

print(len(pairs), Counter(p["type"] for p in pairs).most_common())

In [ ]:
from datasets import Dataset
from vi_fc.prompt import split_prompt_completion
from vi_fc.tools import openai_tools

TOOLS = openai_tools()


def apply(msgs, gen):
    return tokenizer.apply_chat_template(msgs, tools=TOOLS, tokenize=False, add_generation_prompt=gen,
                                         enable_thinking=False)


def render(p):
    prompt, chosen = split_prompt_completion(apply, p["messages"] + [p["chosen"]])
    _, rejected = split_prompt_completion(apply, p["messages"] + [p["rejected"]])
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}


ds = Dataset.from_list([render(p) for p in pairs]).shuffle(seed=0)
print(ds[0]["chosen"]); print("---"); print(ds[0]["rejected"])

In [ ]:
from trl import DPOConfig, DPOTrainer

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0,
)
FastLanguageModel.for_training(model)
bf16 = torch.cuda.is_bf16_supported()
args = DPOConfig(
    output_dir="outputs/dpo", beta=0.1, learning_rate=5e-6, num_train_epochs=1, lr_scheduler_type="cosine",
    warmup_ratio=0.05, per_device_train_batch_size=2, gradient_accumulation_steps=4,
    max_length=MAX_LEN, max_prompt_length=MAX_LEN - 256, optim="adamw_8bit", fp16=not bf16, bf16=bf16,
    logging_steps=10, report_to="none", seed=0,
    save_strategy="steps" if IN_COLAB else "no", save_steps=50, save_total_limit=1,  # Colab: checkpoint lên Drive
)
trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=ds, tokenizer=tokenizer)
trainer.train(resume_from_checkpoint=bool(glob.glob("outputs/dpo/checkpoint-*")))

In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "rewards/margins" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([h["step"] for h in hist], [h["rewards/margins"] for h in hist]); ax[0].set_title("reward margin")
ax[1].plot([h["step"] for h in hist], [h["rewards/accuracies"] for h in hist]); ax[1].set_title("reward accuracy")
fig.savefig("dpo_rewards.png", dpi=110, bbox_inches="tight")

## So SFT vs SFT+DPO trên val + bộ viết tay

In [ ]:
FastLanguageModel.for_inference(model)
fc = FunctionCaller("hf", model, tokenizer=tokenizer, guard=False)
val = load_jsonl(f"{DATA}/val.jsonl")[:200]
manual = load_jsonl(find("data/test_manual.jsonl", "thiếu data/test_manual.jsonl của repo"))
for name, recs in (("val", val), ("manual", manual)):
    m = compute_metrics(predict(recs, fc, "sft-dpo", name, f"dpo_{name}.jsonl"))
    print(name, json.dumps({k: v for k, v in m.items() if k != "exact_by_scenario"}, ensure_ascii=False))

## Lưu: adapter DPO (đẩy Hub) + bản merge 16-bit cho notebook 04

In [ ]:
model.save_pretrained("vi-fc-dpo-lora")
tokenizer.save_pretrained("vi-fc-dpo-lora")
model.save_pretrained_merged("dpo-merged", tokenizer, save_method="merged_16bit")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])
    api.upload_folder(folder_path="vi-fc-dpo-lora", repo_id="DuongCodeAI/vi-fc-qwen3-1.7b", path_in_repo="dpo-lora")
!du -sh sft-merged dpo-merged vi-fc-dpo-lora